# DeltaNet Beta / Retention Diagnostics

This notebook records the DeltaNet `beta` values during an in-context fit pass and plots their sequence-position statistics. For l2-normalized DeltaNet keys, the old-state retention in the current key direction is approximately `1 - beta`, so large non-decaying beta values mean the state keeps being overwritten even at long context lengths.

In [ ]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import torch

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "PFNs").exists():
    repo_root = repo_root.parent
if not (repo_root / "PFNs").exists():
    raise RuntimeError("Could not find repo root containing PFNs/.")

sys.path.insert(0, str(repo_root / "PFNs"))

from fla.layers.delta_net import DeltaNet
from pfns.experiments.model_benchmarks.model_registry import MODEL_FAMILIES
from pfns.experiments.model_benchmarks.models import load_models_for_benchmark

warnings.filterwarnings(
    "ignore",
    message="ShortConvolution is crucial to the performance.*",
    category=UserWarning,
)

plt.rcParams["figure.dpi"] = 120


In [ ]:
MODEL_FAMILY = "non_causal_fla"
MODEL_NAME = "Non_Causal_DeltaNet"

TRAIN_CONTEXT_LEN = 128_000
N_TEST = 16
BATCH_SIZE = 1
NUM_FEATURES = 20
PLOT_MAX_POINTS = 4096

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
autocast_dtype = (
    torch.bfloat16
    if device.type == "cuda" and torch.cuda.is_bf16_supported()
    else torch.float16
)
use_autocast = device.type == "cuda"
print(f"device={device}, use_autocast={use_autocast}, autocast_dtype={autocast_dtype}")


In [ ]:
model_registry = MODEL_FAMILIES[MODEL_FAMILY]
model_config = model_registry[MODEL_NAME]

models, configs = load_models_for_benchmark(
    {MODEL_NAME: model_config},
    device=str(device),
)
model = models[MODEL_NAME]
main_config = configs[MODEL_NAME]
model.eval()

print(f"loaded {MODEL_NAME}: {model_config.get('display_name', MODEL_NAME)}")
print(f"wandb_run_id={model_config.get('wandb_run_id')}")
print(f"parameters={sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")


In [ ]:
get_batch = main_config.priors[0].create_get_batch_method()
batch = get_batch(
    batch_size=BATCH_SIZE,
    seq_len=TRAIN_CONTEXT_LEN + N_TEST,
    num_features=NUM_FEATURES,
    single_eval_pos=TRAIN_CONTEXT_LEN,
    n_targets_per_input=1,
)

train_x = batch.x[:, :TRAIN_CONTEXT_LEN].to(device)
train_y = batch.y[:, :TRAIN_CONTEXT_LEN].to(device)
test_x = batch.x[:, TRAIN_CONTEXT_LEN:].to(device)

print(f"train_x={tuple(train_x.shape)}, train_y={tuple(train_y.shape)}, test_x={tuple(test_x.shape)}")


In [ ]:
beta_records = []
handles = []


def make_beta_hook(layer_name: str, layer: DeltaNet):
    def hook(_module, _inputs, output):
        beta = output.detach().float().sigmoid()
        if layer.allow_neg_eigval:
            beta = beta * 2.0
        beta_records.append({"layer": layer_name, "beta": beta.cpu()})

    return hook


for name, layer in model.named_modules():
    if isinstance(layer, DeltaNet) and layer.use_beta:
        handles.append(layer.b_proj.register_forward_hook(make_beta_hook(name, layer)))

print(f"registered {len(handles)} DeltaNet beta hooks")

try:
    with torch.no_grad(), torch.autocast(
        device_type=device.type,
        dtype=autocast_dtype,
        enabled=use_autocast,
    ):
        state = model.incontext_fit(train_x, train_y)
finally:
    for handle in handles:
        handle.remove()

print(f"recorded {len(beta_records)} layer beta tensors")
for record in beta_records[:3]:
    print(record["layer"], tuple(record["beta"].shape))


In [ ]:
def values_by_position(values: torch.Tensor) -> torch.Tensor:
    """Return a [seq_len, num_observations] view for beta-like tensors."""
    if values.ndim == 2:
        values = values.unsqueeze(0)
    if values.ndim < 3:
        raise ValueError(f"Expected beta tensor with at least 2 dims, got {values.shape}")
    seq_dim = values.ndim - 2
    seq_len = values.shape[seq_dim]
    return values.movedim(seq_dim, 0).reshape(seq_len, -1)


def summarize_by_position(values: torch.Tensor) -> dict[str, torch.Tensor]:
    flat = values_by_position(values)
    return {
        "mean": flat.mean(dim=1),
        "median": flat.median(dim=1).values,
        "q10": flat.quantile(0.10, dim=1),
        "q90": flat.quantile(0.90, dim=1),
    }


def downsample_summary(
    summary: dict[str, torch.Tensor],
    *,
    max_points: int = PLOT_MAX_POINTS,
) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
    n_positions = next(iter(summary.values())).numel()
    if n_positions <= max_points:
        idx = torch.arange(n_positions)
    else:
        idx = torch.linspace(0, n_positions - 1, max_points).round().long()
    return idx, {key: value[idx] for key, value in summary.items()}


def plot_position_band(record: dict, *, mode: str = "beta") -> None:
    beta = record["beta"]
    if mode == "beta":
        values = beta
        ylabel = "beta / overwrite strength"
    elif mode == "retention":
        values = 1.0 - beta
        ylabel = "1 - beta / key-direction retention"
    else:
        raise ValueError(mode)

    summary = summarize_by_position(values)
    x, summary = downsample_summary(summary)
    plt.figure(figsize=(9, 3))
    plt.plot(x, summary["mean"], label="mean")
    plt.plot(x, summary["median"], label="median", alpha=0.8)
    plt.fill_between(x, summary["q10"], summary["q90"], alpha=0.2, label="10-90%")
    plt.axhline(0.0, color="black", linewidth=0.8, alpha=0.4)
    plt.title(f"{record['layer']} - {mode}")
    plt.xlabel("train sequence position")
    plt.ylabel(ylabel)
    plt.legend()
    plt.tight_layout()
    plt.show()


selected_layer_index = -1
selected_record = beta_records[selected_layer_index]
plot_position_band(selected_record, mode="beta")
plot_position_band(selected_record, mode="retention")


In [ ]:
layer_names = [record["layer"] for record in beta_records]
mean_beta_full = torch.stack([
    summarize_by_position(record["beta"])["mean"]
    for record in beta_records
])
if mean_beta_full.shape[1] > PLOT_MAX_POINTS:
    heatmap_idx = torch.linspace(0, mean_beta_full.shape[1] - 1, PLOT_MAX_POINTS).round().long()
    mean_beta = mean_beta_full[:, heatmap_idx]
else:
    heatmap_idx = torch.arange(mean_beta_full.shape[1])
    mean_beta = mean_beta_full

plt.figure(figsize=(10, max(3, 0.28 * len(layer_names))))
plt.imshow(
    mean_beta,
    aspect="auto",
    interpolation="nearest",
    origin="lower",
    extent=[int(heatmap_idx[0]), int(heatmap_idx[-1]), -0.5, len(layer_names) - 0.5],
)
plt.colorbar(label="mean beta")
plt.yticks(range(len(layer_names)), layer_names, fontsize=7)
plt.xlabel("train sequence position")
plt.ylabel("DeltaNet layer")
plt.title("Mean DeltaNet beta by layer and position")
plt.tight_layout()
plt.show()


In [ ]:
# Diagnostic for repeated overwrite. Very negative values mean old information
# along repeatedly hit key directions would be rapidly attenuated.
beta_flat = values_by_position(selected_record["beta"])
retention_flat = (1.0 - beta_flat).abs().clamp_min(1e-6)
cum_log_abs_retention = retention_flat.log().cumsum(dim=0)

summary = {
    "mean": cum_log_abs_retention.mean(dim=1),
    "q10": cum_log_abs_retention.quantile(0.10, dim=1),
    "q90": cum_log_abs_retention.quantile(0.90, dim=1),
}
x, summary = downsample_summary(summary)

plt.figure(figsize=(9, 3))
plt.plot(x, summary["mean"], label="mean")
plt.fill_between(x, summary["q10"], summary["q90"], alpha=0.2, label="10-90%")
plt.title(f"{selected_record['layer']} - cumulative log(abs(1 - beta))")
plt.xlabel("train sequence position")
plt.ylabel("cumulative log retention")
plt.legend()
plt.tight_layout()
plt.show()
